# Data Cleaning

In [86]:
import pandas as pd
import numpy as np

data = {
    "driver": ["VER", "ver", "VER", "HAM", "HAM", "HAM", "VER"],
    "lap_number": [1, 2, 3, 1, 2, 3, 4],
    "lap_time_s": [90.1, None, 91.2, 90.8, 89.9, 90.3, 88.5],
    "compound": ["SOFT", "SOFT", "soft", "MEDIUM", "MEDIUM", "MEDIUM", "SOFT"],
    "speed_kmh": [287.3, 291.1, 450.0, 288.4, 290.1, 289.7, 293.2],
    "is_pit_lap": [False, False, False, False, False, False, True]
}

df_raw = pd.DataFrame(data)



### Data diagnosis

In [87]:
print(df_raw.shape)
print(df_raw.dtypes)
print(df_raw.head())
print(df_raw.describe())
print(df_raw.isnull().sum())
print(df_raw["compound"].unique())
print(df_raw["driver"].unique())

(7, 6)
driver            str
lap_number      int64
lap_time_s    float64
compound          str
speed_kmh     float64
is_pit_lap       bool
dtype: object
  driver  lap_number  lap_time_s compound  speed_kmh  is_pit_lap
0    VER           1        90.1     SOFT      287.3       False
1    ver           2         NaN     SOFT      291.1       False
2    VER           3        91.2     soft      450.0       False
3    HAM           1        90.8   MEDIUM      288.4       False
4    HAM           2        89.9   MEDIUM      290.1       False
       lap_number  lap_time_s   speed_kmh
count    7.000000    6.000000    7.000000
mean     2.285714   90.133333  312.828571
std      1.112697    0.930949   60.516381
min      1.000000   88.500000  287.300000
25%      1.500000   89.950000  289.050000
50%      2.000000   90.200000  290.100000
75%      3.000000   90.675000  292.150000
max      4.000000   91.200000  450.000000
driver        0
lap_number    0
lap_time_s    1
compound      0
speed_kmh     0

- There are typos of "soft" and "ver" which are counted double. The possible problem resources is various data entries.
- Probably there is a hardware problem which filled a row in lap_time_s "None" value.

### Standardizing Typos

In [88]:
# Standardize compound names
compound_map = {
    "soft": "SOFT", "Soft": "SOFT", "S": "SOFT",
    "medium": "MEDIUM", "Medium": "MEDIUM", "M": "MEDIUM",
    "hard": "HARD", "Hard": "HARD", "H": "HARD"
}
df_raw["compound"] = df_raw["compound"].str.strip().map(compound_map).fillna(df_raw["compound"])

driver_map = {
    "ver": "VER", "Ver": "VER", "V": "VER",
    "ham": "HAM", "Ham": "HAM", "H": "HAM"
}
df_raw["driver"] = df_raw["driver"].str.strip().map(driver_map).fillna(df_raw["driver"])
print(df_raw["compound"].unique())
print(df_raw["driver"].unique())
assert set(df_raw["compound"].unique()).issubset({"SOFT", "MEDIUM", "HARD"}), "Expected only SOFT, MEDIUM, HARD values in compound column"
assert set(df_raw["driver"].unique()).issubset({"VER", "HAM"}), "Expected only VER, HAM values in driver column"

<StringArray>
['SOFT', 'MEDIUM']
Length: 2, dtype: str
<StringArray>
['VER', 'HAM']
Length: 2, dtype: str


### Impossible speed values handling

In [89]:
assert len(df_raw[df_raw["speed_kmh"] >= 300]) == 1, \
    f"Expected exactly 1 impossible speed, found: {df_raw[df_raw['speed_kmh'] >= 300]['speed_kmh'].tolist()}"


# In real case this would be a data quality issue (should be re-exported), but for this exercise we will coerce the value to NaN and handle it later.
df_clean = df_raw.copy()
df_clean.loc[df_clean["speed_kmh"] >= 300, "speed_kmh"] = np.nan
print(df_clean)

  driver  lap_number  lap_time_s compound  speed_kmh  is_pit_lap
0    VER           1        90.1     SOFT      287.3       False
1    VER           2         NaN     SOFT      291.1       False
2    VER           3        91.2     SOFT        NaN       False
3    HAM           1        90.8   MEDIUM      288.4       False
4    HAM           2        89.9   MEDIUM      290.1       False
5    HAM           3        90.3   MEDIUM      289.7       False
6    VER           4        88.5     SOFT      293.2        True


- The source of the previous problem is an export issue leading to high and impossible numbers.

### Missing lap handling

In [90]:
assert df_clean["lap_time_s"].isnull().sum() == 1, "Expected exactly 1 NaN in lap_time_s column"

df_clean["lap_time_s"] = pd.to_numeric(df_clean["lap_time_s"], errors="coerce")
print(df_clean)

  driver  lap_number  lap_time_s compound  speed_kmh  is_pit_lap
0    VER           1        90.1     SOFT      287.3       False
1    VER           2         NaN     SOFT      291.1       False
2    VER           3        91.2     SOFT        NaN       False
3    HAM           1        90.8   MEDIUM      288.4       False
4    HAM           2        89.9   MEDIUM      290.1       False
5    HAM           3        90.3   MEDIUM      289.7       False
6    VER           4        88.5     SOFT      293.2        True


- The Hardware problem caused "None" value in the 2nd lap of "VER" and we have to drop it later in order to analyse the valid laps.

### Creating the ready-to-analyse DataFrame

In [91]:
df_clean["is_valid_lap"] = (
    (df_clean["is_pit_lap"] == False) &
    (df_clean["lap_time_s"].notnull()) &
    (df_clean["speed_kmh"].notnull())
)
df_analysis = df_clean[df_clean["is_valid_lap"]].copy()
print(df_analysis)

  driver  lap_number  lap_time_s compound  speed_kmh  is_pit_lap  is_valid_lap
0    VER           1        90.1     SOFT      287.3       False          True
3    HAM           1        90.8   MEDIUM      288.4       False          True
4    HAM           2        89.9   MEDIUM      290.1       False          True
5    HAM           3        90.3   MEDIUM      289.7       False          True


### "snake_case" renaming and MyChron-style name cleaning

In [92]:
# "snake_case" column names for consistency
rename_map = {
    "laptime": "lap_time_s",
    "RPM_val": "rpm",
    "Lat": "gps_lat_deg",
    "Lon": "gps_lon_deg",
    "compound_type": "compound",
    "SPEED": "speed_kmh",
    "BrakeTemp": "brake_temp_c",
    "driver_name": "driver",
    "LapNum": "lap_number",
}

df_messy = pd.DataFrame(columns=["laptime", "RPM_val", "Lat", "Lon", 
                                   "compound_type", "SPEED", "BrakeTemp", 
                                   "driver_name", "LapNum"])
df_renamed = df_messy.rename(columns=rename_map)
assert not any(col in df_renamed.columns for col in rename_map.keys()), \
    "Some columns were not renamed"
print(df_renamed.columns.tolist())

# MyChron columns cleaning

messy_columns = [
    "Speed (km/h)", "RPM (1/min)", "GPS Latitude (deg)", 
    "GPS Longitude (deg)", "Brake Temp (C)", "Lap Time (s)"
]
import sys
sys.path.append("../scripts")
from lap_utils import clean_mychron_columns

messy_df = pd.DataFrame(columns=messy_columns)
cleaned_df = clean_mychron_columns(messy_df)
print(cleaned_df.columns.tolist())


['lap_time_s', 'rpm', 'gps_lat_deg', 'gps_lon_deg', 'compound', 'speed_kmh', 'brake_temp_c', 'driver', 'lap_number']
['speed_kmh', 'rpm', 'gps_latitude_deg', 'gps_longitude_deg', 'brake_temp_c', 'lap_time_s']


### Interpolation and Drop decision

In [93]:
speed_data = {
    "speed_kmh": [287.3, 291.1, None, None, None, 289.7, 293.2, 290.4]}
lap_time_data = {
    "lap_number": [1, 2, 3, 4, 5, 6, 7, 8],
    "lap_time_s": [90.1, 90.4, None, None, 91.2, 90.8, 90.6, 90.9]
}

df_speed = pd.DataFrame(speed_data)
df_lap_time = pd.DataFrame(lap_time_data)


df_speed_clean = df_speed.copy()
df_lap_time_clean = df_lap_time.copy()



df_lap_time_clean["lap_time_s"] = pd.to_numeric(df_lap_time_clean["lap_time_s"], errors="coerce")
df_lap_time_clean = df_lap_time_clean[df_lap_time_clean["lap_time_s"].notnull()]



df_speed_clean["speed_kmh"] = pd.to_numeric(df_speed_clean["speed_kmh"], errors="coerce")
df_speed_clean = df_speed_clean.interpolate(method="linear", limit_direction="both")

print(df_speed_clean)
print(df_lap_time_clean)

   speed_kmh
0     287.30
1     291.10
2     290.75
3     290.40
4     290.05
5     289.70
6     293.20
7     290.40
   lap_number  lap_time_s
0           1        90.1
1           2        90.4
4           5        91.2
5           6        90.8
6           7        90.6
7           8        90.9
